# Movie Review Sentiment Analysis

In [ ]:
# ===============================
# STEP 1: Import required libraries
# ===============================
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_dataset
import numpy as np
import torch

# ===============================
# STEP 2: Load Dataset
# ===============================
# We are using a ready-made dataset of movie reviews (IMDB)
dataset = load_dataset("imdb")

# let's use small data to make training faster
small_train = dataset["train"].shuffle(seed=42).select(range(500))
small_test = dataset["test"].shuffle(seed=42).select(range(1000))

# ===============================
# STEP 3: Load Tokenizer
# ===============================
# Tokenizer converts text into numbers (tokens)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Function to tokenize text
def tokenize_function(example):
    return tokenizer(
        example["text"],           # actual review text
        padding="max_length",      # make all inputs same length
        truncation=True            # cut long text if needed
    )

# Apply tokenizer on dataset
tokenized_train = small_train.map(tokenize_function, batched=True)
tokenized_test = small_test.map(tokenize_function, batched=True)

# ===============================
# STEP 4: Load Model
# ===============================
# Pretrained Transformer model for classification
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2   # Positive or Negative
)

# ===============================
# STEP 5: Training Configuration
# ===============================
training_args = TrainingArguments(
    output_dir="./results",          # where model is saved
    eval_strategy="epoch",     # evaluate after each epoch
    learning_rate=2e-5,              # small learning rate for transformers
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to="none"
)

# ===============================
# STEP 6: Evaluation Metric
# ===============================
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Get predicted class
    predictions = np.argmax(logits, axis=1)

    # Calculate accuracy
    accuracy = (predictions == labels).mean()

    return {"accuracy": accuracy}

# ===============================
# STEP 7: Trainer Object
# ===============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# ===============================
# STEP 8: Train the Model
# ===============================
trainer.train()

# ===============================
# STEP 9: Evaluate Model
# ===============================
results = trainer.evaluate()
print("Evaluation Results:", results)

# ===============================
# STEP 10: Make Predictions
# ===============================
def predict_review(text):
    # Convert text to tokens
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    # Get model output
    outputs = model(**inputs)

    # Convert logits to prediction
    logits = outputs.logits
    predicted_class_id = torch.argmax(logits).item()

    # Map result to label
    if predicted_class_id == 1:
        return "Positive 😊"
    else:
        return "Negative 😞"

# ===============================
# TEST THE MODEL
# ===============================
print(predict_review("This movie was absolutely amazing! I loved it."))
print(predict_review("Worst movie ever. Waste of time."))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.628450,0.752000


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.628450,0.752000


In [ ]:
pip install hf_transfer

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"